In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
import numpy as np
import pandas as pd
from PIL import Image
import os

In [2]:
base_image_dir = "/home/jhong.dongyou/Desktop/orange_smile/data/tanlikesmath/diabetic-retinopathy-resized/versions/7"
df = pd.read_csv(os.path.join(base_image_dir, 'trainLabels_cropped.csv'))
df['path'] = df['image'].map(lambda x: os.path.join(base_image_dir,'resized_train_cropped/resized_train_cropped','{}.jpeg'.format(x)))
df = df.drop(columns=['image'])
df = df.sample(frac=1).reset_index(drop=True) #shuffle dataframe
df['level'] = (df['level'] > 1).astype(int) # Disease or no disease
df.head(10)

,Unnamed: 0.1,Unnamed: 0,level,path
0,3013,3016,0,/home/jhong.dongyou/Desktop/blue_smile/neurflo...
1,28623,28639,0,/home/jhong.dongyou/Desktop/blue_smile/neurflo...
2,29543,29559,0,/home/jhong.dongyou/Desktop/blue_smile/neurflo...
3,15159,15169,0,/home/jhong.dongyou/Desktop/blue_smile/neurflo...
4,2697,2700,0,/home/jhong.dongyou/Desktop/blue_smile/neurflo...
5,2498,2501,0,/home/jhong.dongyou/Desktop/blue_smile/neurflo...
6,13170,13179,1,/home/jhong.dongyou/Desktop/blue_smile/neurflo...
7,21425,21439,1,/home/jhong.dongyou/Desktop/blue_smile/neurflo...
8,6345,6349,0,/home/jhong.dongyou/Desktop/blue_smile/neurflo...
9,11083,11090,0,/home/jhong.dongyou/Desktop/blue_smile/neurflo...


In [3]:
from torch.utils.data import Dataset
from PIL import Image

class DiabeticRetinopathyDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['path']
        label = self.df.iloc[idx]['level']
        
        image = Image.open(img_path).convert("RGB") 
        image = image.resize((224, 224))
        
        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.long)

    # Load dataset
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(360),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
from sklearn.model_selection import train_test_split
train_dataset, val_dataset = train_test_split(df, test_size=0.2)

train_dataset = DiabeticRetinopathyDataset(train_dataset, transform=transform)
val_dataset = DiabeticRetinopathyDataset(val_dataset, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

In [10]:
# Define model
model = models.resnet50(pretrained=True)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)  # Binary classification

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

In [11]:
for batch in train_loader:
    print("Batch loaded!")
    break  # load one batch，make sure DataLoader works


Batch loaded!


In [12]:
from tqdm import tqdm
# Training loop
epochs = 16


for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    model.train()
    running_loss = 0.0

    progress_bar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Training Epoch {epoch+1}")
    for i, (inputs, labels) in progress_bar:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

        # update bar
        progress_bar.set_postfix(loss=running_loss / (i + 1))

    print(f"Epoch {epoch+1} completed, Average Loss: {running_loss/len(train_loader):.4f}")

# Evaluation
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
        y_pred.extend(probs)
        y_true.extend(labels.cpu().numpy())

auc = roc_auc_score(y_true, y_pred)
print("AUC:", auc)

# Save model in torchvision-compatible format
torch.save(model.state_dict(), 'resnet50_diabetic_retinopathy.pth')

Epoch 1/16


Training Epoch 1: 100%|██████████| 1756/1756 [10:44<00:00,  2.73it/s, loss=0.373]


Epoch 1 completed, Average Loss: 0.3734
Epoch 2/16


Training Epoch 2: 100%|██████████| 1756/1756 [10:41<00:00,  2.74it/s, loss=0.332]


Epoch 2 completed, Average Loss: 0.3324
Epoch 3/16


Training Epoch 3: 100%|██████████| 1756/1756 [10:34<00:00,  2.77it/s, loss=0.312]


... (23 outputs left out)

Training Epoch 15: 100%|██████████| 1756/1756 [10:36<00:00,  2.76it/s, loss=0.242]


Epoch 15 completed, Average Loss: 0.2425
Epoch 16/16


Training Epoch 16: 100%|██████████| 1756/1756 [10:38<00:00,  2.75it/s, loss=0.241]


Epoch 16 completed, Average Loss: 0.2415
AUC: 0.9208150772011118
